In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import pandas as pd
from pathlib import Path

In [4]:
df1= pd.read_csv(r"D:\Michael\Behavior\C3\dbnst_endo_pan_c5 - Data Profile - Event Logs.csv")

In [5]:
# ── Mapping: Observation N → r_day + full animal name (C3 cohort) ─────────────
obs_map = pd.DataFrame({
    "obs_num": [ 1,     2,     3,     4,     5,     6,     7,     8,     9,    10,    11,    12],
    "r_day":   [ 1,     1,     1,     1,     1,     1,     2,     2,     2,     2,     2,     2],
    "animal":  ["C3F2","C3M2","C3F3","C3M3","C3F4","C3M4","C3F2","C3M2","C3F3","C3M3","C3F4","C3M4"],
})

# ── Load Event Log (C3) ───────────────────────────────────────────────────────
df1 = pd.read_csv(r"D:\Michael\Behavior\C3\dbnst_endo_pan_c5 - Data Profile - Event Logs.csv")

df1["obs_num"] = df1["Observation"].str.extract(r"(\d+)$").astype(int)
df1 = df1.merge(obs_map, on="obs_num", how="left")
df1["session"] = "R" + df1["r_day"].astype(str)
df1 = df1.drop(columns=["obs_num", "r_day"])

# ── Load C1/C2 xlsx files ─────────────────────────────────────────────────────
base_dir = Path(r"D:\Michael\Behavior")

dfs = []
for subdir in ["C1", "C2"]:
    for fpath in (base_dir / subdir).glob("*.[cx][sl][vs]*"):
        parts = fpath.stem.split(" - ")
        df = pd.read_excel(fpath) if fpath.suffix == ".xlsx" else pd.read_csv(fpath)
        df["animal"] = parts[-2]   # e.g. C1F2, C2M3
        df["session"] = parts[-1]  # e.g. R1, R2
        dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

# ── Stack all cohorts ─────────────────────────────────────────────────────────
all_data = pd.concat([combined, df1], ignore_index=True)


In [6]:

all_data

,Date_Time_Absolute_dmy_hmsf,Date_dmy,Time_Absolute_hms,Time_Absolute_f,Time_Relative_hmsf,Time_Relative_hms,Time_Relative_f,Time_Relative_sf,Duration_sf,Observation,Event_Log,Behavior,Event_Type,Comment,animal,session,Result_Container
0,2024-10-01 11:03:49.989000,2024-10-01 11:03:49.989000,2024-10-01 11:03:49,989,0 days 00:02:08.523000,0 days 00:02:08,523,128.523,0.800352,Observation 4,Event log,Escape,State start,NaN,C1F2,R1,NaN
1,2024-10-01 11:03:50.789000,2024-10-01 11:03:50.789000,2024-10-01 11:03:50,789,0 days 00:02:09.324000,0 days 00:02:09,324,129.324,0.000000,Observation 4,Event log,Escape,State stop,NaN,C1F2,R1,NaN
2,2024-10-01 11:03:52.656000,2024-10-01 11:03:52.656000,2024-10-01 11:03:52,656,0 days 00:02:11.191000,0 days 00:02:11,191,131.191,0.900396,Observation 4,Event log,Tail Sniffing,State start,NaN,C1F2,R1,NaN
3,2024-10-01 11:03:53.557000,2024-10-01 11:03:53.557000,2024-10-01 11:03:53,557,0 days 00:02:12.091000,0 days 00:02:12,91,132.091,0.000000,Observation 4,Event log,Tail Sniffing,State stop,NaN,C1F2,R1,NaN
4,2024-10-01 11:03:54.357000,2024-10-01 11:03:54.357000,2024-10-01 11:03:54,357,0 days 00:02:12.892000,0 days 00:02:12,892,132.892,0.933744,Observation 4,Event log,Tail Sniffing,State start,NaN,C1F2,R1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8533,27-02-2026 09:19:40.836,27-02-2026,09:19:40,836,00:12:16.457,00:12:16,457,736.457,0.000000,Observation 9,Event log,Face Sniffing,State stop,NaN,C3F3,R2,Results
8534,27-02-2026 09:19:41.569,27-02-2026,09:19:41,569,00:12:17.191,00:12:17,191,737.191,2.153110,Observation 9,Event log,Digging,State start,NaN,C3F3,R2,Results
8535,27-02-2026 09:19:42.970,27-02-2026,09:19:42,970,00:12:18.592,00:12:18,592,738.592,0.000000,Observation 9,Event log,NaN,State point,NaN,C3F3,R2,Results
8536,27-02-2026 09:19:43.689,27-02-2026,09:19:43,689,00:12:19.311,00:12:19,311,739.311,0.000000,Observation 9,Event log,NaN,State point,Summary: mostly prosocial behavior and digging...,C3F3,R2,Results


In [7]:
df = all_data[['animal','session','Time_Relative_hmsf','Behavior','Event_Type','Duration_sf','Observation']]

In [8]:
df['time']=df['Time_Relative_hmsf'].apply(pd.to_timedelta)
df

,animal,session,Time_Relative_hmsf,Behavior,Event_Type,Duration_sf,Observation,time
0,C1F2,R1,0 days 00:02:08.523000,Escape,State start,0.800352,Observation 4,0 days 00:02:08.523000
1,C1F2,R1,0 days 00:02:09.324000,Escape,State stop,0.000000,Observation 4,0 days 00:02:09.324000
2,C1F2,R1,0 days 00:02:11.191000,Tail Sniffing,State start,0.900396,Observation 4,0 days 00:02:11.191000
3,C1F2,R1,0 days 00:02:12.091000,Tail Sniffing,State stop,0.000000,Observation 4,0 days 00:02:12.091000
4,C1F2,R1,0 days 00:02:12.892000,Tail Sniffing,State start,0.933744,Observation 4,0 days 00:02:12.892000
...,...,...,...,...,...,...,...,...
8533,C3F3,R2,00:12:16.457,Face Sniffing,State stop,0.000000,Observation 9,0 days 00:12:16.457000
8534,C3F3,R2,00:12:17.191,Digging,State start,2.153110,Observation 9,0 days 00:12:17.191000
8535,C3F3,R2,00:12:18.592,NaN,State point,0.000000,Observation 9,0 days 00:12:18.592000
8536,C3F3,R2,00:12:19.311,NaN,State point,0.000000,Observation 9,0 days 00:12:19.311000


In [10]:
df.animal.unique()

<StringArray>
['C1F2', 'C1M1', 'C1M2', 'C2F1', 'C2M4', 'C3F2', 'C3M3', 'C3F4', 'C3M4',
 'C3M2', 'C3F3']
Length: 11, dtype: str

In [12]:
def get_intervals(group):
    starts = group[group["Event_Type"] == "State start"].reset_index(drop=True)
    stops  = group[group["Event_Type"] == "State stop"].reset_index(drop=True)

    return pd.DataFrame({
        "Behavior": starts["Behavior"],
        "Interval": pd.IntervalIndex.from_arrays(
                        starts["time"].dt.total_seconds(),
                        stops["time"].dt.total_seconds(),
                        closed="both"),}).assign(
                            start=lambda df: df["Interval"].apply(lambda x: x.left),
                            end=lambda df:   df["Interval"].apply(lambda x: x.right),
                            ).sort_values("start").reset_index(drop=True)

interval_dict = {obs: get_intervals(grp)
    for obs, grp in df.groupby("Observation")}

observations = df["Observation"].unique()
interval_dict = {obs: get_intervals(grp)
    for obs, grp in df.groupby("Observation")}


In [13]:
overlaps_all = {obs: find_overlapping_intervals(grp)for obs, grp in interval_dict.items()}
overlaps_all

{'Observation 1':                Behavior            Interval    start      end
 67              Digging  [307.068, 308.536]  307.068  308.536
 68  Anogenital Sniffing  [307.102, 309.436]  307.102  309.436,
 'Observation 10':                             Behavior            Interval    start      end
 72                            Attack   [371.13, 379.267]  371.130  379.267
 73  White Mouse Behavior of Interest  [378.433, 381.001]  378.433  381.001
 74                     Tail-Rattling  [379.567, 381.268]  379.567  381.268,
 'Observation 11':                           Behavior            Interval    start      end
 34  Thigmotaxis (without freezing)  [215.695, 218.963]  215.695  218.963
 35                   Self-Grooming   [218.529, 220.13]  218.529  220.130
 36                   Self-Grooming  [219.797, 221.231]  219.797  221.231,
 'Observation 12':                            Behavior            Interval    start      end
 17   Thigmotaxis (without freezing)  [176.844, 177.678]  176.

In [14]:
def find_overlapping_intervals(intervals):
    intervals["start"] = intervals["Interval"].apply(lambda x: x.left)
    intervals["end"]   = intervals["Interval"].apply(lambda x: x.right)

    # Sort by start time (important)
    intervals = intervals.sort_values("start").reset_index(drop=True)

    # Find overlaps
    overlap_mask = intervals["start"] < intervals["end"].shift(1)
    overlapping = intervals.loc[overlap_mask | overlap_mask.shift(-1).fillna(False)]

    return overlapping

def merge_and_clip_intervals(overlapping):
    intervals = overlapping.copy().sort_values("start").reset_index(drop=True)
    result = []
    i = 0
    while i < len(intervals):
        current = intervals.iloc[i].copy()
        j = i + 1
        while j < len(intervals):
            nxt = intervals.iloc[j]
            if nxt["start"] >= current["end"]:
                break
            if nxt["Behavior"] == current["Behavior"]:
                current["end"] = max(current["end"], nxt["end"])
                intervals.at[j, "start"] = current["end"]
                j += 1
            else:
                current["end"] = nxt["start"]
                break
        result.append(current)
        i = j if (j > i + 1) else i + 1
    out = pd.DataFrame(result).reset_index(drop=True)
    out["Interval"] = pd.arrays.IntervalArray.from_arrays(
        out["start"], out["end"], closed="both")
    return out[["Behavior", "Interval", "start", "end"]]



# Plot Behavior Rasters

In [11]:
behaviors = df["Behavior"].unique()
behaviors

<StringArray>
[                          'Escape',                    'Tail Sniffing',
                    'Face Sniffing',                    'Body Sniffing',
              'Anogenital Sniffing',                                nan,
                          'Chasing',                         'Mounting',
                    'Self-Grooming',   'Thigmotaxis (without freezing)',
       'Other Behavior of Interest',                          'Digging',
                         'Grooming',                'Freezing (social)',
            'Freezing (non-social)',                           'Attack',
           'White Mouse Aggression',                    'Cord Grabbing',
               'Aggression (Misc.)',         'Social Avoidance (Misc.)',
      'Aggressive Grooming/Barbing',                    'Tail-Rattling',
 'White Mouse Behavior of Interest',            'Non-Prosocial (Misc.)']
Length: 24, dtype: str

In [12]:
# ── Sub-behavior color map (colors corrected to match categories) ─────────────
color_map_subtypes = {
    # Aggressive — reds/oranges
    'Attack':                           '#C62828',
    'Aggressive Grooming/Barbing':      '#E53935',
    'Tail-Rattling':                    '#FF5722',
    'White Mouse Aggression':           '#BF360C',
    'Mounting':                         '#EF6C00',  # fixed: was teal
    'Aggression (Misc.)':               '#FF7043',

    # Pro-Social — greens/teals
    'Tail Sniffing':                    '#2E7D32',
    'Body Sniffing':                    '#43A047',
    'Anogenital Sniffing':              '#66BB6A',
    'Face Sniffing':                    '#26A69A',
    'Grooming':                         '#00897B',

    # Social Avoidance — purples
    'Freezing (social)':                '#6A1B9A',  # fixed: was yellow
    'Freezing (non-social)':            '#8E24AA',  # fixed: was amber
    'Social Avoidance (Misc.)':         '#9C27B0',  # fixed: was blue-gray
    'Escape':                           '#7B1FA2',  # fixed: was indigo
    'Digging':                          '#AB47BC',  # fixed: was soft indigo
    'Thigmotaxis (without freezing)':   '#7E57C2',

    # Other — grays
    'White Mouse Behavior of Interest': '#444042',
    'Non-Prosocial (Misc.)':            '#90A4AE',
    'Self-Grooming':                    '#757575',  # fixed: was purple
    'Cord Grabbing':                    '#BDBDBD',  # fixed: was purple
}

# ── 4-category color map ──────────────────────────────────────────────────────
color_map_categories = {
    'Aggressive':        "#C54A4A",
    'Pro-Social':        "#57A35B",
    'Social Avoidance':  '#7B1FA2',
    'Other':             '#78909C',
}

# ── Classification function ───────────────────────────────────────────────────
_aggressive = {
    'Attack', 'Aggressive Grooming/Barbing', 'Tail-Rattling',
    'White Mouse Aggression', 'Mounting', 'Aggression (Misc.)',
}
_prosocial = {
    'Tail Sniffing', 'Body Sniffing', 'Anogenital Sniffing',
    'Face Sniffing', 'Grooming',
}
_avoidance = {
    'Freezing (social)', 'Freezing (non-social)', 'Social Avoidance (Misc.)',
    'Escape', 'Digging', 'Thigmotaxis (without freezing)',
}

def classify_behavior(behavior: str) -> str:
    if behavior in _aggressive:
        return 'Aggressive'
    if behavior in _prosocial:
        return 'Pro-Social'
    if behavior in _avoidance:
        return 'Social Avoidance'
    return 'Other'

In [34]:
df['behavior_type']=df['Behavior'].apply(classify_behavior)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf
import pandas as pd
from pathlib import Path

def build_intervals(session_df: pd.DataFrame, 
                    group_col: str = "Behavior") -> pd.DataFrame:
    intervals = []
    for value, grp in session_df.groupby(group_col):
        starts = grp[grp["Event_Type"] == "State start"]["time"].dt.total_seconds().to_numpy()
        stops  = grp[grp["Event_Type"] == "State stop" ]["time"].dt.total_seconds().to_numpy()
        for s, e in zip(starts, stops):
            intervals.append({group_col: value, "start": s, "end": e})
    return pd.DataFrame(intervals) if intervals else pd.DataFrame(columns=[group_col, "start", "end"])


def plot_animal_timeline(animal: str,
                         df: pd.DataFrame,
                         color_map: dict,
                         output_dir: str | Path,
                         group_col: str = "Behavior",
                         filename_suffix: str = "") -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    animal_df = df[df["animal"] == animal]

    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

    for ax, session in zip(axes, ["R1", "R2"]):
        session_df = animal_df[animal_df["session"] == session]
        intervals  = build_intervals(session_df, group_col=group_col)

        seen_labels = set()
        for _, row in intervals.iterrows():
            label = row[group_col] if row[group_col] not in seen_labels else ""
            seen_labels.add(row[group_col])

            ax.barh(
                y=0,
                width=row["end"] - row["start"],
                left=row["start"],
                height=0.8,
                alpha=0.6,
                color=color_map.get(row[group_col], "#aaaaaa"),
                label=label,
            )

        ax.set_ylim([-0.5, 0.5])
        ax.set_xlim([0, 700])
        ax.vlines(120, -0.5, 0.5, color="r", linestyles="dashed", linewidth=1)
        ax.set_ylabel(session, fontsize=10, rotation=0, labelpad=40, va="center")
        ax.set_yticks([])
        ax.spines[["top", "right", "left"]].set_visible(False)

        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(
                handles, labels,
                title=group_col,
                frameon=False,
                loc="center left",
                bbox_to_anchor=(1.01, 0.5),
                fontsize=8,
            )

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Behavior Timeline — {animal}", y=1.01)
    plt.tight_layout()

    fname = f"{animal}{('_' + filename_suffix) if filename_suffix else ''}.pdf"
    fig.savefig(output_dir / fname, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {output_dir / fname}")

In [38]:
# ── Run for every animal ──────────────────────────────────────────────────────
OUTPUT_DIR = r"D:\Michael\Behavior\Plots"
for animal in df["animal"].unique():
    plot_animal_timeline(animal, df, color_map_subtypes, OUTPUT_DIR,group_col='Behavior')
    plot_animal_timeline(animal, df, color_map_categories, OUTPUT_DIR,group_col='behavior_type',filename_suffix='categories')

Saved: D:\Michael\Behavior\Plots\C1F2.pdf
Saved: D:\Michael\Behavior\Plots\C1F2_categories.pdf
Saved: D:\Michael\Behavior\Plots\C1M1.pdf
Saved: D:\Michael\Behavior\Plots\C1M1_categories.pdf
Saved: D:\Michael\Behavior\Plots\C1M2.pdf
Saved: D:\Michael\Behavior\Plots\C1M2_categories.pdf
Saved: D:\Michael\Behavior\Plots\C2F1.pdf
Saved: D:\Michael\Behavior\Plots\C2F1_categories.pdf
Saved: D:\Michael\Behavior\Plots\C2M4.pdf
Saved: D:\Michael\Behavior\Plots\C2M4_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3F2.pdf
Saved: D:\Michael\Behavior\Plots\C3F2_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3M3.pdf
Saved: D:\Michael\Behavior\Plots\C3M3_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3F4.pdf
Saved: D:\Michael\Behavior\Plots\C3F4_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3M4.pdf
Saved: D:\Michael\Behavior\Plots\C3M4_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3M2.pdf
Saved: D:\Michael\Behavior\Plots\C3M2_categories.pdf
Saved: D:\Michael\Behavior\Plots\C3F3.pdf
Saved: D

In [42]:
df.columns

Index(['animal', 'session', 'Time_Relative_hmsf', 'Behavior', 'Event_Type',
       'Duration_sf', 'Observation', 'time', 'behavior_type'],
      dtype='str')

In [43]:
df2 = df.drop(['Duration_sf','Observation','Time_Relative_hmsf'],axis=1)
df2

,animal,session,Behavior,Event_Type,time,behavior_type
0,C1F2,R1,Escape,State start,0 days 00:02:08.523000,Social Avoidance
1,C1F2,R1,Escape,State stop,0 days 00:02:09.324000,Social Avoidance
2,C1F2,R1,Tail Sniffing,State start,0 days 00:02:11.191000,Pro-Social
3,C1F2,R1,Tail Sniffing,State stop,0 days 00:02:12.091000,Pro-Social
4,C1F2,R1,Tail Sniffing,State start,0 days 00:02:12.892000,Pro-Social
...,...,...,...,...,...,...
8533,C3F3,R2,Face Sniffing,State stop,0 days 00:12:16.457000,Pro-Social
8534,C3F3,R2,Digging,State start,0 days 00:12:17.191000,Social Avoidance
8535,C3F3,R2,NaN,State point,0 days 00:12:18.592000,Other
8536,C3F3,R2,NaN,State point,0 days 00:12:19.311000,Other


In [44]:
df2.to_csv(r"D:\Michael\Behavior\intervals_allmice.csv")